In [1]:

import numpy as np
import pandas as pd
import pickle
from numba import njit
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt
from scipy.stats import sem

# Load existing results
existing_results = pd.read_csv('ldh_kappa_perturbation_results.csv')

print("Existing results loaded:")
print(f"Total rows: {len(existing_results)}")
print(f"\nKappa values present: {sorted(existing_results['kappa'].unique())}")
print(f"\nColumns: {existing_results.columns.tolist()}")
print("\nFirst few rows:")
print(existing_results.head())

# Load omega values
with open('omega_values_N1e6.pkl', 'rb') as f:
 omega_values = pickle.load(f)
 
print(f"\nOmega values loaded. Shape: {omega_values.shape}")
print(f"Values range from {omega_values.min()} to {omega_values.max()}")


Existing results loaded:
Total rows: 40

Kappa values present: [0.23408, 0.33408]

Columns: ['t', 'magnitude', 'red_S2_pct', 'red_S3_pct', 'difference_pct', 'kappa', 'perturbation']

First few rows:
 t magnitude red_S2_pct red_S3_pct difference_pct kappa \
0 1.798362e+06 10.929853 46.953045 32.062279 14.890766 0.33408 
1 1.932311e+06 10.853255 53.437039 32.140485 21.296554 0.33408 
2 1.862496e+06 8.398880 80.218519 49.892183 30.326337 0.33408 
3 1.190101e+06 8.457154 39.774238 28.127551 11.646688 0.33408 
4 1.257351e+06 9.552555 65.356017 63.087162 2.268856 0.33408 

 perturbation 
0 0.05 
1 0.05 
2 0.05 
3 0.05 
4 0.05 

Omega values loaded. Shape: (1000000,)
Values range from 0 to 19


In [2]:

# Define the analysis plan
print("="*80)
print("ANALYSIS PLAN")
print("="*80)
print()
print("Objective: Map S₂-dominance strength as a function of κ for the L_DH function")
print()
print("Steps:")
print("1. Define L_DH coefficient function with flexible κ parameter")
print("2. Implement Numba-accelerated Dirichlet sum evaluation with ω-class decomposition")
print("3. For each κ in {0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4}:")
print(" a. Find top 10 peaks via coarse grid search + local optimization")
print(" b. At each peak, perform causal perturbation analysis (S₂ and S₃)")
print(" c. Calculate mean percentage reduction in |D_F| for each perturbation")
print("4. Combine with existing data for κ ≈ 0.234 and κ ≈ 0.334")
print("5. For each κ, compute mean S₂-S₃ reduction difference and SEM")
print("6. Plot S₂-dominance strength vs. κ with error bars")
print("7. Identify if relationship is monotonic and locate κ_optimal")
print()
print("Statistical methods:")
print("- Mean and standard error of the mean (SEM) for reduction differences")
print("- Visual inspection of trend with error bars")
print()
print("Computational constraints:")
print("- N = 10⁶ (from existing artifacts and analysis history)")
print("- t range: [10⁶, 2·10⁶]")
print("- Coarse grid: 10,000 points for initial peak search")
print("- Kahan summation for final peak evaluation and optimization")
print()
print("="*80)


ANALYSIS PLAN

Objective: Map S₂-dominance strength as a function of κ for the L_DH function

Steps:
1. Define L_DH coefficient function with flexible κ parameter
2. Implement Numba-accelerated Dirichlet sum evaluation with ω-class decomposition
3. For each κ in {0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4}:
 a. Find top 10 peaks via coarse grid search + local optimization
 b. At each peak, perform causal perturbation analysis (S₂ and S₃)
 c. Calculate mean percentage reduction in |D_F| for each perturbation
4. Combine with existing data for κ ≈ 0.234 and κ ≈ 0.334
5. For each κ, compute mean S₂-S₃ reduction difference and SEM
6. Plot S₂-dominance strength vs. κ with error bars
7. Identify if relationship is monotonic and locate κ_optimal

Statistical methods:
- Mean and standard error of the mean (SEM) for reduction differences
- Visual inspection of trend with error bars

Computational constraints:
- N = 10⁶ (from existing artifacts and analysis history)
- t range: [10⁶, 2·10⁶]
- Coarse gri

In [3]:

# Step 1: Define L_DH coefficient function
def generate_ldh_coefficients(N, kappa, seed=42):
 """
 Generate L_DH coefficients: a_n = (-1)^(Ω(n) + [log n > κ log N])
 
 Parameters:
 -----------
 N : int
 Truncation parameter
 kappa : float
 Threshold parameter
 seed : int
 Random seed (not used here but kept for consistency)
 
 Returns:
 --------
 coeffs : ndarray
 Array of coefficients where coeffs[i] corresponds to a_{i+1}
 """
 n_values = np.arange(1, N+1)
 omega = omega_values[:N] # Already loaded, 0-indexed
 
 # Compute threshold
 threshold = kappa * np.log(N)
 log_n = np.log(n_values)
 
 # Indicator function: 1 if log(n) > κ log(N), else 0
 indicator = (log_n > threshold).astype(int)
 
 # a_n = (-1)^(Ω(n) + indicator)
 coeffs = np.power(-1, omega + indicator)
 
 return coeffs

# Test the function
test_coeffs = generate_ldh_coefficients(100, 0.284)
print("L_DH coefficient generation test:")
print(f"First 20 coefficients (κ=0.284): {test_coeffs[:20]}")
print(f"Coefficient distribution: {np.unique(test_coeffs, return_counts=True)}")


L_DH coefficient generation test:
First 20 coefficients (κ=0.284): [ 1 -1 -1 -1 1 -1 1 1 -1 -1 1 1 1 -1 -1 -1 1 1 1 1]
Coefficient distribution: (array([-1, 1]), array([50, 50]))


In [4]:

# Step 2: Implement Numba-accelerated Dirichlet sum evaluation
@njit
def dirichlet_sum_numba(coeffs, t, N):
 """
 Fast Dirichlet sum evaluation using Numba.
 D_F(t; N) = Σ_{n=1}^N a_n / n^(1/2 + it)
 
 Returns complex value.
 """
 result = 0.0 + 0.0j
 for n in range(1, N+1):
 # n^(1/2 + it) = n^(1/2) * n^(it) = sqrt(n) * e^(it log n)
 sqrt_n = np.sqrt(float(n))
 log_n = np.log(float(n))
 phase = t * log_n
 
 # e^(i*phase) = cos(phase) + i*sin(phase)
 real_part = np.cos(phase)
 imag_part = np.sin(phase)
 
 # a_n / n^(1/2+it) = a_n * e^(-i*phase) / sqrt(n)
 contrib_real = coeffs[n-1] * (real_part / sqrt_n)
 contrib_imag = coeffs[n-1] * (-imag_part / sqrt_n)
 
 result += contrib_real + 1j * contrib_imag
 
 return result

# Test Numba function
test_t = 1e6
test_result = dirichlet_sum_numba(test_coeffs, test_t, 100)
print(f"Numba Dirichlet sum test at t={test_t}:")
print(f"Result: {test_result}")
print(f"Magnitude: {np.abs(test_result):.6f}")


Numba Dirichlet sum test at t=1000000.0:
Result: (2.185195995147352-1.2985817899767083j)
Magnitude: 2.541928


In [5]:

# Kahan summation for high-precision evaluation
def dirichlet_sum_kahan(coeffs, t, N):
 """
 High-precision Dirichlet sum using Kahan summation algorithm.
 Returns complex value.
 """
 result_real = 0.0
 result_imag = 0.0
 c_real = 0.0 # Compensation for real part
 c_imag = 0.0 # Compensation for imaginary part
 
 for n in range(1, N+1):
 sqrt_n = np.sqrt(float(n))
 log_n = np.log(float(n))
 phase = t * log_n
 
 real_part = np.cos(phase)
 imag_part = np.sin(phase)
 
 contrib_real = coeffs[n-1] * (real_part / sqrt_n)
 contrib_imag = coeffs[n-1] * (-imag_part / sqrt_n)
 
 # Kahan summation for real part
 y_real = contrib_real - c_real
 t_real = result_real + y_real
 c_real = (t_real - result_real) - y_real
 result_real = t_real
 
 # Kahan summation for imaginary part
 y_imag = contrib_imag - c_imag
 t_imag = result_imag + y_imag
 c_imag = (t_imag - result_imag) - y_imag
 result_imag = t_imag
 
 return result_real + 1j * result_imag

# Test Kahan summation
test_result_kahan = dirichlet_sum_kahan(test_coeffs, test_t, 100)
print(f"Kahan Dirichlet sum test at t={test_t}:")
print(f"Result: {test_result_kahan}")
print(f"Magnitude: {np.abs(test_result_kahan):.6f}")
print(f"Difference from Numba: {np.abs(test_result - test_result_kahan):.2e}")


Kahan Dirichlet sum test at t=1000000.0:
Result: (2.1851959952975113-1.2985817896604324j)
Magnitude: 2.541928
Difference from Numba: 3.50e-10


In [6]:

# Omega-class decomposition functions
def omega_class_decomposition(coeffs, t, N, omega_vals):
 """
 Compute ω-class decomposition: D_F = Σ_k S_k where
 S_k = Σ_{n: Ω(n)=k} a_n / n^(1/2 + it)
 
 Returns dictionary with keys 0, 1, 2, 3, ... containing complex S_k values
 """
 max_omega = int(omega_vals.max())
 S_classes = {k: 0.0 + 0.0j for k in range(max_omega + 1)}
 
 for n in range(1, N+1):
 omega_n = int(omega_vals[n-1])
 
 sqrt_n = np.sqrt(float(n))
 log_n = np.log(float(n))
 phase = t * log_n
 
 real_part = np.cos(phase)
 imag_part = np.sin(phase)
 
 contrib_real = coeffs[n-1] * (real_part / sqrt_n)
 contrib_imag = coeffs[n-1] * (-imag_part / sqrt_n)
 
 S_classes[omega_n] += contrib_real + 1j * contrib_imag
 
 return S_classes

# Test omega-class decomposition
test_decomp = omega_class_decomposition(test_coeffs, test_t, 100, omega_values[:100])
print("Omega-class decomposition test:")
for k in sorted(test_decomp.keys()):
 if np.abs(test_decomp[k]) > 1e-10:
 print(f"S_{k}: {test_decomp[k]:.6f}, |S_{k}|: {np.abs(test_decomp[k]):.6f}")
 
# Verify reconstruction
total_reconstructed = sum(test_decomp.values())
print(f"\nTotal reconstructed: {total_reconstructed}")
print(f"Original sum: {test_result_kahan}")
print(f"Reconstruction error: {np.abs(total_reconstructed - test_result_kahan):.2e}")


Omega-class decomposition test:
S_0: 1.000000+0.000000j, |S_0|: 1.000000
S_1: 0.919707-1.102710j, |S_1|: 1.435908
S_2: 0.223347-0.400195j, |S_2|: 0.458302
S_3: -0.281876-0.023379j, |S_3|: 0.282844
S_4: 0.148418+0.319732j, |S_4|: 0.352500
S_5: 0.124121+0.074026j, |S_5|: 0.144520
S_6: 0.051479-0.166054j, |S_6|: 0.173851

Total reconstructed: (2.1851959952975113-1.2985817896604324j)
Original sum: (2.1851959952975113-1.2985817896604324j)
Reconstruction error: 0.00e+00


In [7]:

# Function to perform causal perturbation analysis
def causal_perturbation_analysis(coeffs, t, N, omega_vals):
 """
 Perform causal perturbation analysis at a single peak location.
 
 Returns:
 --------
 dict with keys:
 - 'magnitude': |D_F| at baseline
 - 'red_S2_pct': percentage reduction when S_2 is multiplied by e^(iπ) = -1
 - 'red_S3_pct': percentage reduction when S_3 is multiplied by e^(iπ) = -1
 - 'difference_pct': red_S2_pct - red_S3_pct
 """
 # Get baseline decomposition
 S_classes = omega_class_decomposition(coeffs, t, N, omega_vals)
 
 # Baseline magnitude
 D_baseline = sum(S_classes.values())
 mag_baseline = np.abs(D_baseline)
 
 # Perturb S_2: multiply by e^(iπ) = -1
 D_S2_perturbed = sum(S_classes[k] * (-1 if k == 2 else 1) for k in S_classes)
 mag_S2_perturbed = np.abs(D_S2_perturbed)
 red_S2_pct = 100 * (mag_baseline - mag_S2_perturbed) / mag_baseline
 
 # Perturb S_3: multiply by e^(iπ) = -1
 D_S3_perturbed = sum(S_classes[k] * (-1 if k == 3 else 1) for k in S_classes)
 mag_S3_perturbed = np.abs(D_S3_perturbed)
 red_S3_pct = 100 * (mag_baseline - mag_S3_perturbed) / mag_baseline
 
 return {
 'magnitude': mag_baseline,
 'red_S2_pct': red_S2_pct,
 'red_S3_pct': red_S3_pct,
 'difference_pct': red_S2_pct - red_S3_pct
 }

# Test perturbation analysis
test_pert = causal_perturbation_analysis(test_coeffs, test_t, 100, omega_values[:100])
print("Causal perturbation analysis test:")
for key, value in test_pert.items():
 print(f"{key}: {value:.4f}")


Causal perturbation analysis test:
magnitude: 2.5419
red_S2_pct: 28.8542
red_S3_pct: -18.8295
difference_pct: 47.6837


In [8]:

# Peak finding functions
def find_peaks_coarse_grid(coeffs, t_min, t_max, n_grid, N):
 """
 Find peaks using coarse grid search with Numba-accelerated evaluation.
 
 Returns array of (t, magnitude) tuples sorted by magnitude (descending).
 """
 t_grid = np.linspace(t_min, t_max, n_grid)
 magnitudes = np.zeros(n_grid)
 
 print(f"Evaluating {n_grid} grid points...")
 for i, t in enumerate(t_grid):
 D = dirichlet_sum_numba(coeffs, t, N)
 magnitudes[i] = np.abs(D)
 
 if (i+1) % 1000 == 0:
 print(f" Progress: {i+1}/{n_grid} points", end='\r')
 
 print(f" Progress: {n_grid}/{n_grid} points")
 
 # Sort by magnitude
 sorted_indices = np.argsort(magnitudes)[::-1]
 peaks = [(t_grid[i], magnitudes[i]) for i in sorted_indices]
 
 return peaks

# Test coarse grid search on small range
print("Testing coarse grid search on small range...")
test_coeffs_large = generate_ldh_coefficients(10000, 0.284)
test_peaks = find_peaks_coarse_grid(test_coeffs_large, 1e6, 1e6 + 10000, 100, 10000)
print(f"\nTop 5 peaks found:")
for i, (t, mag) in enumerate(test_peaks[:5]):
 print(f"{i+1}. t={t:.2f}, |D|={mag:.4f}")


Testing coarse grid search on small range...
Evaluating 100 grid points...
 Progress: 100/100 points

Top 5 peaks found:
1. t=1006868.69, |D|=11.0180
2. t=1000505.05, |D|=9.6731
3. t=1008282.83, |D|=9.1432
4. t=1001212.12, |D|=8.3508
5. t=1003333.33, |D|=8.3157


In [9]:

def refine_peak_location(coeffs, t_init, N, omega_vals, search_radius=1000):
 """
 Refine peak location using scipy.optimize with Kahan summation.
 
 Parameters:
 -----------
 coeffs : array
 Coefficient array
 t_init : float
 Initial t value from coarse grid
 N : int
 Truncation parameter
 omega_vals : array
 Omega values array
 search_radius : float
 Search radius around t_init
 
 Returns:
 --------
 dict with keys: 't', 'magnitude', 'red_S2_pct', 'red_S3_pct', 'difference_pct'
 """
 # Define objective function (negative magnitude for minimization)
 def neg_magnitude(t):
 D = dirichlet_sum_kahan(coeffs, t, N)
 return -np.abs(D)
 
 # Optimize
 result = minimize_scalar(
 neg_magnitude,
 bounds=(t_init - search_radius, t_init + search_radius),
 method='bounded'
 )
 
 t_optimal = result.x
 
 # Perform perturbation analysis at optimal location
 pert_results = causal_perturbation_analysis(coeffs, t_optimal, N, omega_vals)
 pert_results['t'] = t_optimal
 
 return pert_results

# Test peak refinement
print("Testing peak refinement...")
refined_peak = refine_peak_location(test_coeffs_large, test_peaks[0][0], 10000, omega_values[:10000])
print(f"\nRefined peak:")
for key, value in refined_peak.items():
 if key == 't':
 print(f"{key}: {value:.6f}")
 else:
 print(f"{key}: {value:.4f}")


Testing peak refinement...



Refined peak:
magnitude: 18.3465
red_S2_pct: 30.6899
red_S3_pct: 27.3767
difference_pct: 3.3132
t: 1006341.962826


In [10]:

def analyze_kappa_value(kappa, N=1000000, n_grid=10000, n_peaks=10):
 """
 Complete analysis pipeline for a single kappa value:
 1. Generate L_DH coefficients
 2. Find top peaks using coarse grid + refinement
 3. Perform perturbation analysis at each peak
 
 Returns DataFrame with results for all peaks.
 """
 print(f"\n{'='*80}")
 print(f"Analyzing κ = {kappa:.3f}")
 print(f"{'='*80}")
 
 # Generate coefficients
 print(f"Generating L_DH coefficients for N={N}, κ={kappa}...")
 coeffs = generate_ldh_coefficients(N, kappa)
 
 # Coarse grid search
 t_min, t_max = N, 2*N
 print(f"Coarse grid search in t ∈ [{t_min:.0f}, {t_max:.0f}]...")
 coarse_peaks = find_peaks_coarse_grid(coeffs, t_min, t_max, n_grid, N)
 
 print(f"\nTop {n_peaks} coarse peaks:")
 for i in range(min(n_peaks, len(coarse_peaks))):
 t, mag = coarse_peaks[i]
 print(f" {i+1}. t={t:.2f}, |D|={mag:.4f}")
 
 # Refine and analyze peaks
 print(f"\nRefining and analyzing top {n_peaks} peaks...")
 results = []
 for i in range(n_peaks):
 t_init, _ = coarse_peaks[i]
 print(f" Peak {i+1}/{n_peaks}: t_init={t_init:.2f}...", end='')
 
 refined = refine_peak_location(coeffs, t_init, N, omega_values[:N])
 refined['kappa'] = kappa
 results.append(refined)
 
 print(f" t_opt={refined['t']:.2f}, |D|={refined['magnitude']:.4f}, " + 
 f"Δ(S₂-S₃)={refined['difference_pct']:.2f}%")
 
 df = pd.DataFrame(results)
 
 # Summary statistics
 mean_diff = df['difference_pct'].mean()
 sem_diff = sem(df['difference_pct'])
 print(f"\nSummary for κ={kappa:.3f}:")
 print(f" Mean S₂-S₃ difference: {mean_diff:.2f}% ± {sem_diff:.2f}% (SEM)")
 print(f" Range: [{df['difference_pct'].min():.2f}%, {df['difference_pct'].max():.2f}%]")
 
 return df

# Test on κ = 0.25 (should take ~5-10 minutes at N=10^6)
# First, let's do a quick test with smaller N to verify the pipeline works
print("Testing pipeline with N=50000...")
test_df = analyze_kappa_value(0.25, N=50000, n_grid=1000, n_peaks=3)
print("\nTest results:")
print(test_df[['t', 'magnitude', 'red_S2_pct', 'red_S3_pct', 'difference_pct']].head())


Testing pipeline with N=50000...

Analyzing κ = 0.250
Generating L_DH coefficients for N=50000, κ=0.25...
Coarse grid search in t ∈ [50000, 100000]...
Evaluating 1000 grid points...


 Progress: 1000/1000 points

Top 3 coarse peaks:
 1. t=93443.44, |D|=21.0772
 2. t=99299.30, |D|=18.6750
 3. t=78178.18, |D|=15.3183

Refining and analyzing top 3 peaks...
 Peak 1/3: t_init=93443.44...

 t_opt=93207.90, |D|=7.7950, Δ(S₂-S₃)=-12.91%
 Peak 2/3: t_init=99299.30...

 t_opt=99287.48, |D|=5.3317, Δ(S₂-S₃)=-92.63%
 Peak 3/3: t_init=78178.18...

 t_opt=78550.75, |D|=7.2007, Δ(S₂-S₃)=28.01%

Summary for κ=0.250:
 Mean S₂-S₃ difference: -25.84% ± 35.42% (SEM)
 Range: [-92.63%, 28.01%]

Test results:
 t magnitude red_S2_pct red_S3_pct difference_pct
0 93207.901454 7.795020 23.412930 36.321424 -12.908495
1 99287.480427 5.331684 -11.071271 81.559895 -92.631166
2 78550.747298 7.200749 56.462772 28.455147 28.007625


In [11]:

# There seems to be an issue with the numba compilation or the loop.
# Let's debug step by step with very small parameters

print("Debug: Testing with minimal parameters")
print("N=1000, n_grid=100, n_peaks=2")

import time
start = time.time()

kappa_debug = 0.25
coeffs_debug = generate_ldh_coefficients(1000, kappa_debug)
print(f"Coefficients generated: {coeffs_debug.shape}")

t_min, t_max = 1000, 2000
t_grid = np.linspace(t_min, t_max, 100)
print(f"Grid created: {len(t_grid)} points")

# Test a single evaluation
t_test = t_grid[0]
D_test = dirichlet_sum_numba(coeffs_debug, t_test, 1000)
print(f"Single evaluation at t={t_test}: |D|={np.abs(D_test):.4f}")

elapsed = time.time() - start
print(f"Time so far: {elapsed:.2f} seconds")


TimeoutError: Code execution timed out after 1034 seconds